# Framework thinking

An agent framework packages common agent parts and gives them names, defaults, and integrations.

In this notebook, we will:

- see that we already built the main parts;
- understand what a framework adds;
- learn five common framework shapes;
- choose a shape from a real need; and
- know when plain code is enough.

In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import litellm
from dotenv import load_dotenv

course_root = Path.cwd()
if not (course_root / "pyproject.toml").exists():
    course_root = course_root.parent
load_dotenv(course_root / ".env")
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openrouter/deepseek/deepseek-v3.2")
litellm.drop_params = True
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=str(course_root / ".litellm_cache"))

## 1. We already built framework parts

The shoplab package contains the machinery from earlier chapters.

In [ ]:
from shoplab import context, controls, loop, trace, verify
from shoplab.tools import standard_tools

parts = {
    "agent loop": loop.run_agent,
    "verification": verify.check_decision,
    "budget": controls.Budget,
    "approval": controls.require_approval,
    "checkpoint": controls.Checkpoint,
    "trace": trace.Tracer,
    "compaction": context.compact,
    "sub-agent": context.run_subagent,
}

for name, part in parts.items():
    print(name, "->", part.__module__ + "." + part.__name__)

print("tools ->", sorted(standard_tools()))

## 2. What a framework adds

A framework usually adds:

- ready-made control flow;
- integrations with models, tools, and storage;
- standard state objects;
- maintained tracing and retry behavior; and
- examples and a community.

The trade is less direct control. Some behavior moves behind framework defaults.

## 3. Five common shapes

| Shape | Main idea | Part we already built |
|---|---|---|
| Graph | Nodes and edges control the path | Router and tool loop |
| Typed | Input, state, and output have strong schemas | Tool and decision schemas |
| Team | Roles pass work to each other | Supervisor and handoff |
| Model-driven loop | The model chooses each next action | run_agent |
| Code as action | The model writes code that uses tools | A different action space |

## 4. One run can be described in several ways

The same return run has state, a loop, tools, routing choices, and a structured finish. Frameworks emphasize different parts of that same system.

In [ ]:
from shoplab.loop import run_agent
from shoplab.tools import standard_tools

task = (
    "Triage order ORD-7312 for customer CUST-07. The opened Torrent boots were "
    "returned after 18 days. Look up order, customer, and policy, then call finish."
)
system = (
    "Return decision, policy_id, and refund_usd. Do not repeat lookups."
)

path = []
result = run_agent(
    task,
    standard_tools(),
    model=MODEL,
    system=system,
    max_steps=8,
    on_step=lambda step, message: path.append(
        [call.function.name for call in message.tool_calls or []] or ["text"]
    ),
)

print("path:", path)
print("answer:", result.answer)

A graph framework calls each step a node. A model-loop framework calls each step an action. A team framework may call a policy lookup a specialist task. The labels differ; the work is familiar.

## 5. Choose from the requirement

Ask these questions before choosing a library:

1. Must the path be visible and controlled? Choose a graph.
2. Are strict schemas the main need? Choose a typed framework.
3. Do roles need separate tools or ownership? Choose a team framework.
4. Is a simple tool loop enough? Choose a model-driven loop.
5. Does the task benefit from data processing and arithmetic in code? Consider code as action.
6. Must runs pause and survive restarts? Check durable state and human-interrupt support before anything else.

## 6. When plain code is enough

Do not start with a framework when one prompt, a fixed workflow, or a small tool loop solves the problem.

Start with the simplest working design. Add a framework when its defaults and integrations remove real work you would otherwise maintain.

## What to remember

1. A framework packages familiar agent machinery.
2. It buys defaults, integrations, and maintenance.
3. It costs some control and transparency.
4. Most libraries emphasize one of five common shapes.
5. Choose from requirements, not popularity.
6. Plain code is often the right first version.

## Small practice steps

1. Label each step in path as model, tool, or finish.
2. Choose an archetype for a human-approved refund workflow.
3. Name one case where a framework would add more work than it removes.